# Switching Schedulability Plot

This notebook plots the schedulability ratio for the old EDF-VD sufficient test, the new sufficient test, and the EDF-VDSD test from a source CSV with the same switching-data layout.

Change `CSV_PATH` to point at another file with the same columns if you want to reuse the notebook on a different input.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({
    "figure.figsize": (6, 3.5),
    "axes.titlesize": "small",
    "axes.labelsize": "small",
    "legend.fontsize": "small",
    "xtick.labelsize": "small",
    "ytick.labelsize": "small",
})

CSV_PATH = "../outputs/switching.csv"
TEST_COLUMNS = {
    "EDFVD_test": "EDF-VD (sufficient)",
    "EDFVD_test_new": "EDF-VD (new sufficient)",
    "EDFVDSD_test": "EDF-VDSD",
}

def plot_schedulability(csv_path=CSV_PATH, x_col="U"):
    df = pd.read_csv(csv_path)

    required_columns = {x_col, *TEST_COLUMNS.keys()}
    missing_columns = sorted(required_columns.difference(df.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns in {csv_path}: {missing_columns}")

    plot_df = df.loc[:, [x_col, *TEST_COLUMNS.keys()]].rename(columns=TEST_COLUMNS)
    plot_df = plot_df.melt(id_vars=[x_col], var_name="test", value_name="schedulable")
    plot_df["schedulable"] = plot_df["schedulable"].astype(float)

    summary = (
        plot_df.groupby([x_col, "test"], as_index=False)["schedulable"]
        .mean()
        .sort_values([x_col, "test"])
    )

    fig, ax = plt.subplots()
    sns.lineplot(
        data=summary,
        x=x_col,
        y="schedulable",
        hue="test",
        marker="o",
        ax=ax,
    )

    ax.set_xlabel("Average utilisation ($U^*$)")
    ax.set_ylabel("Schedulability ratio")
    ax.set_title("Switching schedulability")
    ax.set_ylim(0, 1.05)
    ax.legend(title="")
    fig.tight_layout()
    return fig, ax, summary

fig, ax, summary = plot_schedulability()
summary

/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/_base.py:949: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/_base.py:949: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  data_subset = grouped_data.get_group(pd_key)
/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/_base.py:949: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in

,U,test,schedulable
0,0.50,EDF-VD (new sufficient),1.0
1,0.50,EDF-VD (sufficient),1.0
2,0.50,EDF-VDSD,1.0
3,0.51,EDF-VD (new sufficient),1.0
4,0.51,EDF-VD (sufficient),1.0
...,...,...,...
145,0.98,EDF-VD (sufficient),0.0
146,0.98,EDF-VDSD,0.0
147,0.99,EDF-VD (new sufficient),0.0
148,0.99,EDF-VD (sufficient),0.0


In [5]:
fig.show()